[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Prepared Statements &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with `plans` and `plans_for`. Run it first. Each task opens
the connections it needs and closes them, so they can be run in any order.


In [1]:
import asyncio
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from asyncpg import exceptions
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def plans(conn):
    """What the server holds for a psycopg connection, without preparing the question itself."""
    return [name for (name,) in conn.execute(                       # prepare=False exempts this one
        "SELECT name FROM pg_prepared_statements ORDER BY name", prepare=False).fetchall()]


async def plans_for(conn):
    """The same for an asyncpg connection, which has no way to exempt the question."""
    return [record["name"] for record in
            await conn.fetch("SELECT name FROM pg_prepared_statements ORDER BY name")]


def against(baseline, measured):
    """A band rather than a number, because a ratio this small is a property of the machine."""
    ratio = baseline / measured
    if ratio < 1.15:
        return "about the same"
    return "a little faster" if ratio < 2 else "much faster"


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


**1.** Six runs, watched.


In [2]:
conn = psycopg.connect("dbname=guide", autocommit=True)

for run in range(1, 7):
    conn.execute("SELECT count(*) FROM events WHERE kind = %s", ("click",))
    print(f"  after run {run}:", plans(conn))

conn.close()


  after run 1: []
  after run 2: []
  after run 3: []
  after run 4: []
  after run 5: []
  after run 6: ['_pg3_0']


Nothing for five runs, then a name on the sixth. `prepare_threshold` is how many runs psycopg wants
to see before it decides the query is worth keeping.


**2.** Prepared at once.


In [3]:
conn = psycopg.connect("dbname=guide", autocommit=True)

conn.execute("SELECT count(*) FROM events WHERE kind = %s", ("click",), prepare=True)
print("after one run:", plans(conn))

conn.close()


after one run: ['_pg3_0']


Worth doing for a query you know will repeat, such as the one statement at the center of a request
handler, and not worth doing for anything built out of a string that changes.


**3.** An asyncpg connection that keeps nothing.


In [4]:
conn = await asyncpg.connect(database="guide", statement_cache_size=0)

for kind in ("click", "view", "purchase"):
    await conn.fetchval("SELECT count(*) FROM events WHERE kind = $1", kind)

print("after three queries:", await plans_for(conn))
await conn.close()


after three queries: []


Empty, and that includes the query that asked. The queries still ran, and every one of them was
parsed and planned from scratch.


**4.** A statement of your own.


In [5]:
conn = await asyncpg.connect(database="guide")

counter = await conn.prepare("SELECT count(*) FROM events WHERE kind = $1")
print("clicks:", await counter.fetchval("click"))
print("views: ", await counter.fetchval("view"))
print("it returns:", [(a.name, a.type.name) for a in counter.get_attributes()])

await conn.close()


clicks: 1666
views:  1667
it returns: [('count', 'int8')]


The same server-side statement, run twice with different parameters. This is what both drivers do
for you underneath, made visible.


**5.** One session cannot see another's.


In [6]:
mine = psycopg.connect("dbname=guide", autocommit=True)
yours = psycopg.connect("dbname=guide", autocommit=True)

mine.execute("SELECT count(*) FROM events", prepare=True)
print("mine sees: ", plans(mine))
print("yours sees:", plans(yours))

mine.close()
yours.close()


mine sees:  ['_pg3_0']
yours sees: []


`pg_prepared_statements` is a view of the asking session and nothing else, which is why closing a
connection needs no cleanup and why a pooler handing out a different connection is a problem.


**6.** Four queries, room for two.


In [7]:
conn = psycopg.connect("dbname=guide", autocommit=True)
conn.prepared_max = 2
conn.prepare_threshold = 0

for number in range(4):
    conn.execute(f"SELECT count(*) + {number} FROM events")
    print(f"  after query {number + 1}:", plans(conn))

conn.close()


  after query 1: ['_pg3_0']
  after query 2: ['_pg3_0', '_pg3_1']
  after query 3: ['_pg3_1', '_pg3_2']
  after query 4: ['_pg3_2', '_pg3_3']


The names keep changing because psycopg discards the least recently used one to make room. Four
different query texts, two plans held, and no query ever benefits from a plan that was thrown away
before it ran again.


---

&#8592; **Back to:** [Prepared Statements](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/12-prepared-statements.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
